In [1]:
import os
import re
from datetime import datetime
import pandas as pd
from bs4 import BeautifulSoup
from itertools import chain
from typing import Iterable
from helpers import get_page_name

In [2]:
csv_filename = "./article_links_06-11-25_20-40-38_with_contents"
articles_df = pd.read_csv(f"{csv_filename}.csv")
articles_df

,link,type,time_ago,headline,content
0,/news/gabriel-bonfim-leveled-up-legend-ufc-veg...,Athletes,4 hours ago,Gabriel Bonfim | Leveled Up By A Legend,"\n\n\n\n\n\n<!DOCTYPE html>\n<html lang=""en-ca..."
1,/news/chris-padilla-blocking-out-the-noise-ufc...,Athletes,5 hours ago,Chris Padilla | Blocking Out The Noise,"\n\n\n\n\n\n<!DOCTYPE html>\n<html lang=""en-ca..."
2,/news/matt-schnell-honest-excited-and-dangerou...,Athletes,5 hours ago,"Matt Schnell | Honest, Excited, And Dangerous","\n\n\n\n\n\n<!DOCTYPE html>\n<html lang=""en-ca..."
3,/news/tecia-pennington-crossroad-ufc-fight-nig...,Athletes,5 hours ago,Tecia Pennington | At A Crossroad,"\n\n\n\n\n\n<!DOCTYPE html>\n<html lang=""en-ca..."
4,/news/uros-medic-back-track-ufc-fight-night-bo...,Athletes,5 hours ago,Uroš Medić Is Back On Track,"\n\n\n\n\n\n<!DOCTYPE html>\n<html lang=""en-ca..."
5,/news/randy-brown-something-bigger-ufc-fight-n...,Athletes,11 hours ago,Randy Brown | Something Bigger,"\n\n\n\n\n\n<!DOCTYPE html>\n<html lang=""en-ca..."
6,/news/joseph-morales-all-way-back-ufc-vegas-11...,Athletes,1 day ago,Joseph Morales | All The Way Back,"\n\n\n\n\n\n<!DOCTYPE html>\n<html lang=""en-ca..."
7,/news/malcolm-wellmaker-built-ground-bantamwei...,Athletes,1 day ago,Malcolm Wellmaker | Built From The Ground Up,"\n\n\n\n\n\n<!DOCTYPE html>\n<html lang=""en-ca..."
8,/news/christian-leroy-duncan-settled-and-start...,Athletes,1 day ago,Christian Leroy Duncan | Settled In And Starti...,"\n\n\n\n\n\n<!DOCTYPE html>\n<html lang=""en-ca..."
9,/news/adrian-yanez-reloaded-ufc-vegas-111,Athletes,3 weeks hence,Adrian Yanez | Reloaded,"\n\n\n\n\n\n<!DOCTYPE html>\n<html lang=""en-ca..."


In [3]:
author_pattern = r"By ([A-Za-z\s.]+).*"
social_media_pattern = r"@([A-Za-z]+)"
text_classes = ["e-p--initial", "field--name-text"]

def get_page_datas(page_contents: Iterable[str]):
    soups = [BeautifulSoup(content, "html.parser") for content in page_contents]

    def get_titles():
        for soup in soups:
            title_element = soup.find("meta", property="og:title")
            yield title_element.get("content") if title_element else None
    
    def get_descriptions():
        for soup in soups:
            description_element = soup.find("meta", property="og:description")
            yield description_element.get("content") if description_element else None
    
    def get_published_times():
        for soup in soups:
            published_time_element = soup.find("meta", property="article:published_time")
            yield datetime.fromisoformat(published_time_element.get("content"))\
                if published_time_element\
                else None
    
    def get_modified_times():
        for soup in soups:
            modified_time_element = soup.find("meta", property="article:modified_time")
            yield datetime.fromisoformat(modified_time_element.get("content"))\
                if modified_time_element\
                else None

    def get_authors():
        for soup in soups:
            credit_element = soup.find("div", "c-hero__article-credit")

            if credit_element:
                credit = credit_element.text.strip().replace("\n", "")
                author_match = re.search(author_pattern, credit)
                yield author_match.group(1).strip() if author_match else None
            else:
                yield None

    def get_social_medias():
        for soup in soups:
            credit_element = soup.find("div", "c-hero__article-credit")

            if credit_element:
                credit = credit_element.text.strip().replace("\n", "")
                social_media_match = re.search(social_media_pattern, credit)
                yield social_media_match.group(1).strip() if social_media_match else None
            else:
                yield None
    
    def get_texts():
        for soup in soups:
            text_containers = list(
                chain.from_iterable(
                    soup.find_all(attrs={"class": text_class}) for text_class in text_classes
                )
            )

            text_elements = list(
                chain.from_iterable(
                    container.find_all("p") for container in text_containers
                )
            )

            non_link_text_elements = [element for element in text_elements
                                      if not element.find_all("a")]

            texts = [element.text.strip() for element in non_link_text_elements]
            combined_text = "\n".join(texts)
            
            yield combined_text

    def get_image_srcs_list():
        # Since encoding a list into a DataFrame can get weird,
        # I will save the image list as a single string, 
        # with each image src on a separate line.
        # Then when downloading images, I can decode this back into a list.
        for soup in soups:
            images_srcs: list[str] = [image.get("src") for image in soup.find_all("img")]
            images_srcs = [
                f"https://ufc.com{src}" if not src.startswith("https") else src
                for src in images_srcs 
            ]

            yield "\n".join(images_srcs)
    
    def get_image_captions():
        for soup in soups:
            image_caption_elements = soup.find_all(attrs={"class": "field--name-image"})
            image_captions = [element.text.strip() for element in image_caption_elements]

            yield "\n".join(image_captions)

    return {
        "title": get_titles(),
        "description": get_descriptions(),
        "published_time": get_published_times(),
        "modified_time": get_modified_times(),
        "author": get_authors(),
        "social_media": get_social_medias(),
        "text": get_texts(),
        "images_srcs": get_image_srcs_list(),
        "images_captions": get_image_captions(),
    }

In [4]:
datas = get_page_datas(articles_df["content"])
for key in datas.keys():
    articles_df[key] = list(datas[key])

articles_df

,link,type,time_ago,headline,content,title,description,published_time,modified_time,author,social_media,text,images_srcs,images_captions
0,/news/gabriel-bonfim-leveled-up-legend-ufc-veg...,Athletes,4 hours ago,Gabriel Bonfim | Leveled Up By A Legend,"\n\n\n\n\n\n<!DOCTYPE html>\n<html lang=""en-ca...",Gabriel Bonfim | Leveled Up By A Legend,Brazilian Welterweight Contender Is Still Grow...,2025-11-06 18:39:27-06:00,2025-11-06 19:05:11-06:00,Simon Head,simonheadsport,It’s often said that fighters learn more from ...,https://ufc.com/images/styles/background_image...,Gabriel Bonfim of Brazil punches Stephen Thomp...
1,/news/chris-padilla-blocking-out-the-noise-ufc...,Athletes,5 hours ago,Chris Padilla | Blocking Out The Noise,"\n\n\n\n\n\n<!DOCTYPE html>\n<html lang=""en-ca...",Chris Padilla | Blocking Out The Noise,Lightweight Prospect Looks To Secure His Fourt...,2025-11-06 17:30:00-06:00,2025-11-06 17:35:33-06:00,Maddyn Johnstone,None,"For Chris Padilla, it doesn’t matter where he ...",https://ufc.com/images/styles/background_image...,Chris Padilla punches Jai Herbert in a lightwe...
2,/news/matt-schnell-honest-excited-and-dangerou...,Athletes,5 hours ago,"Matt Schnell | Honest, Excited, And Dangerous","\n\n\n\n\n\n<!DOCTYPE html>\n<html lang=""en-ca...","Matt Schnell | Honest, Excited, And Dangerous","Veteran Flyweight Discusses Retirement, Return...",2025-11-06 17:14:00-06:00,2025-11-06 17:41:09-06:00,E. Spencer Kyte,spencerkyte,Matt Schnell is an interviewer’s dream; you as...,https://ufc.com/images/styles/background_image...,Matt Schnell elbows Jimmy Flick in a flyweight...
3,/news/tecia-pennington-crossroad-ufc-fight-nig...,Athletes,5 hours ago,Tecia Pennington | At A Crossroad,"\n\n\n\n\n\n<!DOCTYPE html>\n<html lang=""en-ca...",Tecia Pennington | At A Crossroad,"Rankings Mainstay Talks Motherhood, Fighting M...",2025-11-06 16:50:19-06:00,2025-11-06 17:42:01-06:00,E. Spencer Kyte,spencerkyte,Tecia Pennington heads into her fight with Den...,https://ufc.com/images/styles/background_image...,Tecia Pennington kicks Luana Pinheiro of Brazi...
4,/news/uros-medic-back-track-ufc-fight-night-bo...,Athletes,5 hours ago,Uroš Medić Is Back On Track,"\n\n\n\n\n\n<!DOCTYPE html>\n<html lang=""en-ca...",Uroš Medić Is Back On Track,Serbian Striker Has His Sights Set On The Welt...,2025-11-06 13:11:39-06:00,2025-11-06 16:41:33-06:00,Simon Head,simonheadsport,"At the start of 2025, Uroš Medić set himself t...",https://ufc.com/images/styles/background_image...,Uros Medic of Serbia drops Gilbert Urbina with...
5,/news/randy-brown-something-bigger-ufc-fight-n...,Athletes,11 hours ago,Randy Brown | Something Bigger,"\n\n\n\n\n\n<!DOCTYPE html>\n<html lang=""en-ca...",Randy Brown | Something Bigger,"Veteran Talks Main Event Assignment, Providing...",2025-11-06 10:50:18-06:00,2025-11-06 10:34:28-06:00,E. Spencer Kyte,None,Randy Brown has been waiting for this moment f...,https://ufc.com/images/styles/background_image...,Randy Brown of Jamaica knees Nicolas Dalby of ...
6,/news/joseph-morales-all-way-back-ufc-vegas-11...,Athletes,1 day ago,Joseph Morales | All The Way Back,"\n\n\n\n\n\n<!DOCTYPE html>\n<html lang=""en-ca...",Joseph Morales | All The Way Back,Recent TUF Winner Discusses Return To UFC Rost...,2025-11-05 17:10:08-06:00,2025-11-05 17:28:29-06:00,E. Spencer Kyte,spencerkyte,Joseph Morales didn’t really celebrate his vic...,https://ufc.com/images/styles/background_image...,Joseph Morales attempts to submit Roberto Sanc...
7,/news/malcolm-wellmaker-built-ground-bantamwei...,Athletes,1 day ago,Malcolm Wellmaker | Built From The Ground Up,"\n\n\n\n\n\n<!DOCTYPE html>\n<html lang=""en-ca...",Malcolm Wellmaker | Built From The Ground Up,How a Rising Bantamweight Star Turned a Distan...,2025-11-05 11:27:38-06:00,2025-11-05 11:36:50-06:00,Kevin Schuster,None,"Earlier this year, bantamweight Malcolm Wellma...",https://ufc.com/images/styles/background_image...,Malcolm Wellmaker reacts after knocking out Kr...
8,/news/christian-leroy-duncan-se

In [5]:
with_data_filename = f"{csv_filename}_with_data.csv"
articles_df\
    .drop("content", axis=1)\
    .set_index("link")\
    .to_csv(with_data_filename)

print("Saved article metadata and contents to", with_data_filename)

Saved article metadata and contents to ./article_links_06-11-25_20-40-38_with_contents_with_data.csv


In [6]:
new_csv_filename = csv_filename.replace("with_contents", "with_only_metadata")

articles_df\
    .drop("content", axis=1)\
    .drop("text", axis=1)\
    .drop("images_srcs", axis=1)\
    .drop("images_captions", axis=1)\
    .set_index("link")\
    .to_csv(f"{new_csv_filename}.csv")

print("Saved article metadata to", new_csv_filename)

Saved article metadata to ./article_links_06-11-25_20-40-38_with_only_metadata


In [7]:
folder = "./articles"
os.makedirs(folder, exist_ok=True)

for row in articles_df.iloc:
    filename = get_page_name(row["link"])
    
    try:
        with open(f"{folder}/{filename}.txt", "w", encoding="utf-8") as file:
            file.write(row["text"])
    except Exception as e:
        print("Error saving", filename)
        print(e)